# LeetCode 3373 — Maximize the Number of Target Nodes After Connecting Trees II bằng Python

- Link: https://leetcode.com/problems/maximize-the-number-of-target-nodes-after-connecting-trees-ii/
- Độ khó: **Hard**
- Chủ đề: **Cây là đồ thị hai phía**: tô hai màu bằng BFS; "khoảng cách chẵn" ⇔ **cùng màu**; ghép hai cây bằng một cạnh đổi tính chẵn lẻ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxTargetNodes(self, edges1: List[List[int]], edges2: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Nhận ra trong một cây, số cạnh trên đường đi `u → v` **chẵn** ⇔ `u`, `v` **cùng màu** khi tô hai màu (theo độ sâu chẵn/lẻ).
2. Khi nối nút `i` (cây 1) với nút `j` (cây 2): một nút `w` của cây 2 cách `i` một số chẵn cạnh ⇔ `dist(j, w)` **lẻ** ⇔ `w` **khác màu** với `j`.
3. Vì chọn `j` tự do: phần đóng góp của cây 2 = `max(số nút màu 0, số nút màu 1)` — **độc lập với `i`**.
4. Đáp án `answer[i] = (số nút cùng màu với i trong cây 1) + max(c0, c1)` — tổng `O(n + m)`.
5. Kiểm chứng bằng BFS từ mọi nút và vét cạn mọi cạnh nối.

## 2. Đề bài & ví dụ

Hai cây `n` và `m` nút. Nút `u` là **target** của `v` nếu số cạnh trên đường `u → v` chẵn (mỗi nút là target của chính nó). Với mỗi nút `i` của cây 1, được thêm **một** cạnh nối một nút cây 1 với một nút cây 2; trả về số target lớn nhất của `i` (các truy vấn độc lập).

**Ví dụ 1:** → `[8,7,7,8,8]`. **Ví dụ 2:** `edges1` hình sao 5 nút, `edges2` đường 4 nút → `[3,6,6,6,6]`.

**Ràng buộc:** `2 ≤ n, m ≤ 10^5`.

In [1]:
from typing import List
from collections import deque

examples = [
    ([[0, 1], [0, 2], [2, 3], [2, 4]], [[0, 1], [0, 2], [0, 3], [2, 7], [1, 4], [4, 5], [4, 6]], [8, 7, 7, 8, 8]),
    ([[0, 1], [0, 2], [0, 3], [0, 4]], [[0, 1], [1, 2], [2, 3]], [3, 6, 6, 6, 6]),
]
for e1, e2, exp in examples:
    print('n =', len(e1) + 1, 'm =', len(e2) + 1, '-> kỳ vọng', exp)

n = 5 m = 8 -> kỳ vọng [8, 7, 7, 8, 8]
n = 5 m = 4 -> kỳ vọng [3, 6, 6, 6, 6]


## 3. Kiến thức nền

### 3.1. Tô hai màu một cây bằng BFS

In [2]:
def adj(edges):
    n = len(edges) + 1
    g = [[] for _ in range(n)]
    for a, b in edges:
        g[a].append(b); g[b].append(a)
    return g

def two_color(edges):
    g = adj(edges)
    color = [-1] * len(g)
    color[0] = 0
    q = deque([0])
    while q:
        u = q.popleft()
        for v in g[u]:
            if color[v] == -1:
                color[v] = color[u] ^ 1; q.append(v)
    return color

print(two_color(examples[0][0]), two_color(examples[0][1]))

[0, 1, 1, 0, 0] [0, 1, 1, 1, 0, 1, 1, 0]


### 3.2. Tính chẵn lẻ của khoảng cách

`dist(u, v) = depth(u) + depth(v) − 2·depth(lca)` ⇒ `dist(u, v) ≡ depth(u) + depth(v) (mod 2)` ⇒ chẵn ⇔ cùng màu.

### 3.3. Thêm một cạnh giữa hai cây

Đường từ `i` tới `w` (cây 2) là `i → … → (nút nối a ở cây 1) → (nút nối b ở cây 2) → … → w`; số cạnh `= dist₁(i, a) + 1 + dist₂(b, w)`. Để tối đa số `w` có tổng chẵn, chọn `a = i` (không mất gì) và chọn `b` thích hợp.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: cây 1 hình sao tâm 0 — màu: `0` một mình màu 0, các lá màu 1 (4 nút). Cây 2 đường `0-1-2-3`: hai màu mỗi màu 2 nút ⇒ `max = 2`.

- `i = 0`: cùng màu trong cây 1: chỉ `{0}` ⇒ `1 + 2 = 3`.
- `i` là lá: cùng màu `{1,2,3,4}` ⇒ `4 + 2 = 6`.

Khớp `[3, 6, 6, 6, 6]`.

In [3]:
c1, c2 = two_color(examples[1][0]), two_color(examples[1][1])
print('màu cây 1:', c1, '| màu cây 2:', c2, '| max lớp cây 2:', max(c2.count(0), c2.count(1)))

màu cây 1: [0, 1, 1, 1, 1] | màu cây 2: [0, 1, 0, 1] | max lớp cây 2: 2


## 5. Cách 1 — Brute Force: thử mọi cạnh nối, BFS trên cây ghép

In [4]:
def bfs_dist(g, s):
    d = [-1] * len(g); d[s] = 0; q = deque([s])
    while q:
        u = q.popleft()
        for v in g[u]:
            if d[v] == -1:
                d[v] = d[u] + 1; q.append(v)
    return d

def max_target_brute(e1, e2):
    n, m = len(e1) + 1, len(e2) + 1
    ans = []
    for i in range(n):
        best = 0
        for a in range(n):
            for b in range(m):
                g = [[] for _ in range(n + m)]
                for x, y in e1:
                    g[x].append(y); g[y].append(x)
                for x, y in e2:
                    g[n + x].append(n + y); g[n + y].append(n + x)
                g[a].append(n + b); g[n + b].append(a)
                d = bfs_dist(g, i)
                best = max(best, sum(1 for x in d if x % 2 == 0))
        ans.append(best)
    return ans

for e1, e2, exp in examples:
    print(max_target_brute(e1, e2), '| kỳ vọng', exp)

[8, 7, 7, 8, 8] | kỳ vọng [8, 7, 7, 8, 8]
[3, 6, 6, 6, 6] | kỳ vọng [3, 6, 6, 6, 6]


**Độ phức tạp:** `O(n · n · m · (n + m))` — chỉ cho cây rất nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tô màu cây 1: `col1`, đếm `cnt1[0], cnt1[1]`.
2. Tô màu cây 2: `cnt2[0], cnt2[1]`; `best2 = max(cnt2)`.
3. `answer[i] = cnt1[col1[i]] + best2`.

Phần cây 2: nối `i` với `j`, các `w` có `1 + dist₂(j, w)` chẵn ⇔ `w` khác màu `j` ⇒ chọn `j` thuộc lớp **nhỏ hơn** để đếm được lớp **lớn hơn**. Luôn có ít nhất một nút mỗi màu vì `m ≥ 2`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS từ mọi nút (đếm chẵn/lẻ trực tiếp), `O(n² + m²)`

In [5]:
def max_target_allbfs(e1, e2):
    g1, g2 = adj(e1), adj(e2)
    even1 = [sum(1 for x in bfs_dist(g1, i) if x % 2 == 0) for i in range(len(g1))]
    odd2 = [sum(1 for x in bfs_dist(g2, j) if x % 2 == 1) for j in range(len(g2))]
    best2 = max(odd2)
    return [even1[i] + best2 for i in range(len(g1))]

for e1, e2, exp in examples:
    print(max_target_allbfs(e1, e2), '| kỳ vọng', exp)

[8, 7, 7, 8, 8] | kỳ vọng [8, 7, 7, 8, 8]
[3, 6, 6, 6, 6] | kỳ vọng [3, 6, 6, 6, 6]


### 7.2. Cách 3 — Tô hai màu (lời giải chính, `O(n + m)`)

In [6]:
def max_target_color(e1, e2):
    c1, c2 = two_color(e1), two_color(e2)
    cnt1 = [c1.count(0), c1.count(1)]
    best2 = max(c2.count(0), c2.count(1))
    return [cnt1[c] + best2 for c in c1]

for e1, e2, exp in examples:
    print(max_target_color(e1, e2), '| kỳ vọng', exp)

[8, 7, 7, 8, 8] | kỳ vọng [8, 7, 7, 8, 8]
[3, 6, 6, 6, 6] | kỳ vọng [3, 6, 6, 6, 6]


### 7.3. Cách 4 — DP đổi gốc (rerooting) đếm số nút cách chẵn/lẻ

Không dùng tô màu: `down[u] = (#chẵn, #lẻ)` trong cây con của `u` theo khoảng cách từ `u`; đổi gốc: `full[v] = down[v] + swap(full[u] − swap(down[v]))`. Tổng quát hơn (ví dụ cho bài "khoảng cách ≤ k" ở bản I), ở đây dùng để kiểm chứng.

In [7]:
def even_odd_reroot(edges):
    g = adj(edges); n = len(g)
    parent, order = [-1] * n, [0]
    seen = [False] * n; seen[0] = True
    for u in order:
        for v in g[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; order.append(v)
    down = [[1, 0] for _ in range(n)]
    for u in reversed(order):
        p = parent[u]
        if p != -1:
            down[p][0] += down[u][1]; down[p][1] += down[u][0]
    full = [None] * n
    full[0] = down[0][:]
    for u in order[1:]:
        p = parent[u]
        # phần ngoài cây con u, nhìn từ p: full[p] trừ đóng góp của u (đã đổi chẵn/lẻ)
        out_e = full[p][0] - down[u][1]; out_o = full[p][1] - down[u][0]
        full[u] = [down[u][0] + out_o, down[u][1] + out_e]
    return full

def max_target_reroot(e1, e2):
    f1, f2 = even_odd_reroot(e1), even_odd_reroot(e2)
    best2 = max(o for _, o in f2)
    return [e + best2 for e, _ in f1]

for e1, e2, exp in examples:
    print(max_target_reroot(e1, e2), '| kỳ vọng', exp)

[8, 7, 7, 8, 8] | kỳ vọng [8, 7, 7, 8, 8]
[3, 6, 6, 6, 6] | kỳ vọng [3, 6, 6, 6, 6]


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1.

In [8]:
def trace(e1, e2):
    c1, c2 = two_color(e1), two_color(e2)
    cnt1 = [c1.count(0), c1.count(1)]; cnt2 = [c2.count(0), c2.count(1)]
    print('cây 1 màu:', c1, '-> số nút mỗi màu', cnt1)
    print('cây 2 màu:', c2, '-> số nút mỗi màu', cnt2, '-> đóng góp tốt nhất', max(cnt2))
    for i, c in enumerate(c1):
        print(f"i = {i}: cùng màu {c} có {cnt1[c]} nút + {max(cnt2)} = {cnt1[c] + max(cnt2)}")

trace(*examples[0][:2])

cây 1 màu: [0, 1, 1, 0, 0] -> số nút mỗi màu [3, 2]
cây 2 màu: [0, 1, 1, 1, 0, 1, 1, 0] -> số nút mỗi màu [3, 5] -> đóng góp tốt nhất 5
i = 0: cùng màu 0 có 3 nút + 5 = 8
i = 1: cùng màu 1 có 2 nút + 5 = 7
i = 2: cùng màu 1 có 2 nút + 5 = 7
i = 3: cùng màu 0 có 3 nút + 5 = 8
i = 4: cùng màu 0 có 3 nút + 5 = 8


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maxTargetNodes(self, edges1: List[List[int]], edges2: List[List[int]]) -> List[int]:
        def color(edges):
            n = len(edges) + 1
            g = [[] for _ in range(n)]
            for a, b in edges:
                g[a].append(b)
                g[b].append(a)
            col = [-1] * n
            col[0] = 0
            stack = [0]
            while stack:
                u = stack.pop()
                cu = col[u] ^ 1
                for v in g[u]:
                    if col[v] == -1:
                        col[v] = cu
                        stack.append(v)
            return col

        c1 = color(edges1)
        c2 = color(edges2)
        ones1 = sum(c1)
        cnt1 = [len(c1) - ones1, ones1]
        ones2 = sum(c2)
        best2 = max(ones2, len(c2) - ones2)
        return [cnt1[c] + best2 for c in c1]

for e1, e2, exp in examples:
    got = Solution().maxTargetNodes(e1, e2)
    print(got, 'OK' if got == exp else 'SAI')

[8, 7, 7, 8, 8] OK
[3, 6, 6, 6, 6] OK


## 10. Vì sao thuật toán đúng

1. **Trong cây 1:** nút `w` của cây 1 là target của `i` ⇔ `dist₁(i, w)` chẵn (cạnh nối không tạo đường ngắn hơn trong cây 1 vì đồ thị sau khi nối vẫn là cây) ⇔ cùng màu (3.2).
2. **Trong cây 2:** đường duy nhất từ `i` tới `w` đi qua cạnh nối `(a, b)`: độ dài `dist₁(i, a) + 1 + dist₂(b, w)`. Chọn `a = i` (khoảng cách 0); với `a` khác chỉ thay đổi tính chẵn lẻ một hằng số, tương đương chọn `b` khác màu — không tốt hơn được. Số `w` thỏa = số nút khác màu `b` ⇒ tối đa `max(cnt2)`.
3. Hai phần độc lập ⇒ cộng lại.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cạnh nối | `O(n² m (n + m))` | `O(n + m)` |
| 2. BFS từ mọi nút | `O(n² + m²)` | `O(n + m)` |
| 3. Tô hai màu | `O(n + m)` | `O(n + m)` |
| 4. Đổi gốc | `O(n + m)` | `O(n + m)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'allbfs': max_target_allbfs, 'color': max_target_color, 'reroot': max_target_reroot,
         'submit': lambda a, b: Solution().maxTargetNodes(a, b)}
edge = [([[0, 1]], [[0, 1]], [2, 2]), ([[0, 1], [1, 2]], [[0, 1]], [3, 2, 3]), ([[1, 0]], [[0, 1], [0, 2], [0, 3]], [4, 4])]
for e1, e2, exp in examples + edge:
    assert max_target_brute(e1, e2) == exp, (e1, e2, max_target_brute(e1, e2))
    for name, f in impls.items():
        assert f(e1, e2) == exp, (name, e1, e2)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random

def random_tree(n, rng):
    perm = list(range(n)); rng.shuffle(perm)
    return [[perm[v], perm[rng.randrange(v)]] for v in range(1, n)]

rng = random.Random(3373)
cnt = 0
for _ in range(120):
    e1, e2 = random_tree(rng.randint(2, 6), rng), random_tree(rng.randint(2, 6), rng)
    exp = max_target_brute(e1, e2)
    for name, f in impls.items():
        assert f(e1, e2) == exp, (name, e1, e2)
    cnt += 1
for _ in range(200):
    e1, e2 = random_tree(rng.randint(2, 60), rng), random_tree(rng.randint(2, 60), rng)
    assert max_target_allbfs(e1, e2) == max_target_color(e1, e2) == max_target_reroot(e1, e2)
print('Khớp vét cạn trên', cnt, 'cặp cây nhỏ; 3 cách khớp nhau trên 200 cặp cây vừa.')

Khớp vét cạn trên 120 cặp cây nhỏ; 3 cách khớp nhau trên 200 cặp cây vừa.


### 12.3. Ca lớn `n = m = 10^5`

In [12]:
import time
n = 10 ** 5
cases = {
    'đường dài': ([[i, i + 1] for i in range(n - 1)], [[i, i + 1] for i in range(n - 1)]),
    'sao': ([[0, i] for i in range(1, n)], [[0, i] for i in range(1, n)]),
    'ngẫu nhiên': (random_tree(n, rng), random_tree(n, rng)),
}
for name, (e1, e2) in cases.items():
    t = time.time(); r = Solution().maxTargetNodes(e1, e2); t1 = time.time() - t
    t = time.time(); r2 = max_target_reroot(e1, e2); t2 = time.time() - t
    assert r == r2
    print(f"{name:12s}: max = {max(r)}, min = {min(r)} | tô màu {t1:.2f}s | đổi gốc {t2:.2f}s")

đường dài   : max = 100000, min = 100000 | tô màu 0.15s | đổi gốc 0.40s


sao         : max = 199998, min = 100000 | tô màu 0.15s | đổi gốc 0.39s


ngẫu nhiên  : max = 100086, min = 100082 | tô màu 0.30s | đổi gốc 0.78s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng lớp **cùng màu** với nút nối ở cây 2 (quên cạnh nối đổi tính chẵn lẻ)

In [13]:
def wrong_same_color_tree2(e1, e2):
    c1, c2 = two_color(e1), two_color(e2)
    cnt1 = [c1.count(0), c1.count(1)]
    return [cnt1[c] + c2.count(c2[0]) for c in c1]       # SAI: cố định nút nối 0 và đếm cùng màu

e1, e2, _ = examples[0]
print('sai :', wrong_same_color_tree2(e1, e2), '| đúng:', Solution().maxTargetNodes(e1, e2))

sai : [6, 5, 5, 6, 6] | đúng: [8, 7, 7, 8, 8]


**Vì sao sai:** cạnh nối thêm 1 vào độ dài; nút cây 2 là target khi **khác màu** với nút nối. Hơn nữa phải chọn nút nối tốt nhất (lấy `max`).

### Lỗi 2 — Dùng lớp chứa nút 0 của cây 1 cho mọi `i`

In [14]:
def wrong_fixed_class(e1, e2):
    c1, c2 = two_color(e1), two_color(e2)
    k = c1.count(c1[0])
    return [k + max(c2.count(0), c2.count(1)) for _ in c1]

e1, e2, _ = examples[1]
print('sai :', wrong_fixed_class(e1, e2), '| đúng:', Solution().maxTargetNodes(e1, e2))

sai : [3, 3, 3, 3, 3] | đúng: [3, 6, 6, 6, 6]


**Vì sao sai:** mỗi `i` đếm lớp **của chính nó**.

### Lỗi 3 — Coi target là "khoảng cách ≤ k" (nhầm với bản I) hoặc đếm cả nút cách lẻ

In [15]:
def wrong_all_nodes(e1, e2):
    n = len(e1) + 1; c2 = two_color(e2)
    return [n + max(c2.count(0), c2.count(1)) for _ in range(n)]

print('sai :', wrong_all_nodes(e1, e2), '| đúng:', Solution().maxTargetNodes(e1, e2))

sai : [7, 7, 7, 7, 7] | đúng: [3, 6, 6, 6, 6]


**Vì sao sai:** trong cây 1 chỉ các nút cách `i` một số **chẵn** cạnh mới là target.

### Lỗi 4 — DFS đệ quy tô màu trên đường dài

In [16]:
def color_recursive(edges):
    g = adj(edges); col = [-1] * len(g)
    def dfs(u, c):
        col[u] = c
        for v in g[u]:
            if col[v] == -1:
                dfs(v, c ^ 1)
    dfs(0, 0)
    return col

try:
    color_recursive(cases['đường dài'][0])
except RecursionError as ex:
    print('RecursionError:', ex)

RecursionError: maximum recursion depth exceeded


**Vì sao sai:** đường `10^5` nút vượt giới hạn đệ quy. Dùng BFS/ngăn xếp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Khoảng cách chẵn/lẻ trên cây" ⇒ **tô hai màu** (cây là đồ thị hai phía).
- Thêm một cạnh giữa hai cây ⇒ đổi tính chẵn lẻ một lần; phần cây kia độc lập với truy vấn ⇒ tính một lần.
- Khi đề hỏi "tối đa với mỗi `i`" nhưng một phần không phụ thuộc `i` ⇒ tách ra tính trước.
- Đổi gốc (rerooting) là công cụ tổng quát khi điều kiện phức tạp hơn chẵn/lẻ.

## 15. Biến thể của bài

1. **LC 3372 — bản I:** target khi khoảng cách `≤ k` ⇒ BFS giới hạn từ mỗi nút (`n, m ≤ 1000`).
2. Target khi khoảng cách chia hết cho 3 ⇒ tô 3 "màu" theo `depth mod 3` **không** còn đúng (khoảng cách mod 3 không xác định bởi depth mod 3) ⇒ cần đổi gốc với 3 bộ đếm.
3. Nối `k` cây thành chuỗi ⇒ mỗi cạnh nối đổi tính chẵn lẻ.
4. Đếm số cặp `(i, w)` target trong một cây ⇒ `c0² + c1²`.

In [17]:
# Biến thể 2: số nút cách u một khoảng ≡ r (mod 3), bằng đổi gốc với 3 bộ đếm
def mod3_reroot(edges):
    g = adj(edges); n = len(g)
    parent, order = [-1] * n, [0]; seen = [False] * n; seen[0] = True
    for u in order:
        for v in g[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; order.append(v)
    down = [[1, 0, 0] for _ in range(n)]
    for u in reversed(order):
        p = parent[u]
        if p != -1:
            for r in range(3):
                down[p][(r + 1) % 3] += down[u][r]
    full = [None] * n; full[0] = down[0][:]
    for u in order[1:]:
        p = parent[u]
        out = [full[p][r] - down[u][(r - 1) % 3] for r in range(3)]     # phần ngoài cây con u, nhìn từ p
        full[u] = [down[u][r] + out[(r - 1) % 3] for r in range(3)]
    return full

rng = random.Random(15)
for _ in range(100):
    e = random_tree(rng.randint(2, 30), rng); g = adj(e)
    f = mod3_reroot(e)
    for u in range(len(g)):
        d = bfs_dist(g, u)
        assert f[u] == [sum(1 for x in d if x % 3 == r) for r in range(3)]
print('Biến thể mod 3 (đổi gốc) khớp BFS.')

Biến thể mod 3 (đổi gốc) khớp BFS.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi cạnh nối | rất lớn | Oracle |
| 2 | BFS mọi nút | `O(n² + m²)` | Trung gian |
| 3 | Tô hai màu | `O(n + m)` | **Nộp bài** |
| 4 | Đổi gốc | `O(n + m)` | Tổng quát hơn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** cả hai cây là cạnh đơn `[[0,1]]`.

**Bài 2 (tính tay):** cây 1 đường 3 nút, cây 2 sao 4 nút.

**Bài 3 (điền TODO):** với mỗi `i` trả về luôn **nút `j` của cây 2** nên nối.

**Bài 4 (tự viết lại):** LC 3372 — target khi khoảng cách `≤ k` (BFS giới hạn).

**Bài 5 (mini challenge):** biến thể 4 — số cặp `(u, v)` (có thứ tự, kể cả `u = v`) có khoảng cách chẵn trong một cây.

In [18]:
def best_partner_todo(e1, e2):
    # TODO
    raise NotImplementedError

def max_target_k_todo(e1, e2, k):
    # TODO
    raise NotImplementedError

def even_pairs_todo(edges):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mỗi cây hai màu, mỗi màu 1 nút. (2) `1 + 1`. (3) `[2, 2]`.

**Bài 2.** (1) Đường 3 nút: màu `0,1,0`. (2) Sao 4 nút: tâm 1, lá 3 ⇒ max 3. (3) `[5, 4, 5]`.

**Bài 3.** (1) Cần `j` thuộc lớp **nhỏ hơn** của cây 2. (2) Bất kỳ nút nào của lớp đó. (3) Kiểm tra bằng BFS.

**Bài 4.** (1) Đếm nút cách `≤ k` từ mỗi `i` ở cây 1. (2) Cây 2: `≤ k − 1` từ nút nối, lấy max. (3) `k = 0` ⇒ chỉ chính `i`.

**Bài 5.** (1) Cặp cùng màu. (2) `c0² + c1²`. (3) Vét cạn BFS kiểm tra.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().maxTargetNodes([[0, 1]], [[0, 1]]) == [2, 2]
assert Solution().maxTargetNodes([[0, 1], [1, 2]], [[0, 1], [0, 2], [0, 3]]) == [5, 4, 5]

def best_partner(e1, e2):
    c2 = two_color(e2)
    small = 0 if c2.count(0) <= c2.count(1) else 1        # nối vào lớp nhỏ để đếm lớp lớn
    j = c2.index(small)
    return [(Solution().maxTargetNodes(e1, e2)[i], j) for i in range(len(e1) + 1)]

e1, e2, _ = examples[0]
n = len(e1) + 1
for (val, j) in best_partner(e1, e2)[:1]:
    g = [[] for _ in range(n + len(e2) + 1)]
    for x, y in e1: g[x].append(y); g[y].append(x)
    for x, y in e2: g[n + x].append(n + y); g[n + y].append(n + x)
    g[0].append(n + j); g[n + j].append(0)
    assert sum(1 for d in bfs_dist(g, 0) if d % 2 == 0) == val
print('Bài 1: [2, 2] | Bài 2: [5, 4, 5] | Bài 3 ví dụ 1:', best_partner(e1, e2))

Bài 1: [2, 2] | Bài 2: [5, 4, 5] | Bài 3 ví dụ 1: [(8, 0), (7, 0), (7, 0), (8, 0), (8, 0)]


In [20]:
# Bài 4, 5
def max_target_k(e1, e2, k):
    g1, g2 = adj(e1), adj(e2)
    def within(g, s, lim):
        if lim < 0:
            return 0
        return sum(1 for d in bfs_dist(g, s) if d <= lim)
    best2 = max(within(g2, j, k - 1) for j in range(len(g2)))
    return [within(g1, i, k) + best2 for i in range(len(g1))]

assert max_target_k([[0, 1], [0, 2], [2, 3], [2, 4]], [[0, 1], [0, 2], [0, 3], [2, 7], [1, 4], [4, 5], [4, 6]], 2) == [9, 7, 9, 8, 8]
assert max_target_k([[0, 1], [0, 2], [0, 3], [0, 4]], [[0, 1], [1, 2], [2, 3]], 1) == [6, 3, 3, 3, 3]     # ví dụ LC 3372

def even_pairs(edges):
    c = two_color(edges)
    return c.count(0) ** 2 + c.count(1) ** 2

rng = random.Random(5)
for _ in range(100):
    e = random_tree(rng.randint(2, 25), rng); g = adj(e)
    assert even_pairs(e) == sum(1 for u in range(len(g)) for d in bfs_dist(g, u) if d % 2 == 0)
print('Bài 4: ví dụ LC 3372 đúng | Bài 5 khớp BFS.')

Bài 4: ví dụ LC 3372 đúng | Bài 5 khớp BFS.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
c1 = tô_màu(cây 1); c2 = tô_màu(cây 2)          # BFS/ngăn xếp, không đệ quy
cnt1 = [số nút màu 0, số nút màu 1] của cây 1
best2 = max(số nút màu 0, màu 1) của cây 2
answer[i] = cnt1[c1[i]] + best2
```

**Checklist:**

- [ ] Khoảng cách chẵn ⇔ cùng màu?
- [ ] Cạnh nối đổi tính chẵn lẻ ⇒ lấy `max` của hai lớp cây 2?
- [ ] Mỗi `i` dùng lớp của chính nó trong cây 1?
- [ ] Không đệ quy trên cây sâu `10^5`?
- [ ] `best2` tính **một lần**?